# CIFAR-10 image classification model

## Step 1: Import Libraries 

In [1]:
#Check tensorflow and numpy version
import tensorflow as tf
print(tf.__version__)
import numpy as np
print(np.__version__)

2026-01-13 13:16:49.164002: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 13:16:49.253894: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 13:16:50.777794: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2026-01-13 13:

2.11.0
1.26.4


In [2]:
# basic
import numpy as np
import pandas as pd
import keras

# visuals
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report

# keras
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import load_model
from keras.losses import CategoricalCrossentropy
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import models, layers, regularizers
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv2D, Flatten
from tensorflow.keras.layers import ZeroPadding2D, MaxPool2D, GlobalAveragePooling2D, LeakyReLU
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers.experimental import AdamW
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger

# Warnings
import warnings
warnings.filterwarnings('ignore')


## Step 2: Importing Cifar-10 data  

CIFAR-10 images are 32×32×3 (RGB). Although grayscale reduces channel count
and computation, color information is essential for distinguishing many
CIFAR-10 classes, for instance frog vs cat has strong differences in color distribution.

In [3]:
# Load CIFAR-10 Dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
print("Original shapes:")
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)

# Class Names
class_names = [
    "airplane", "automobile", "bird", "cat", "deer", 
    "dog", "frog", "horse", "ship", "truck"]
num_classes = 10

# Neutralize the dataset
y_train = np_utils.to_categorical(y_train, num_classes)
y_test = np_utils.to_categorical(y_test, num_classes)

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

Original shapes:
x_train: (50000, 32, 32, 3)
x_test: (10000, 32, 32, 3)


## Step 3: Data Augmentation

Data augmentation is used in CIFAR-10 to reduce overfitting caused by the dataset’s small size and low image resolution. It artificially increases training diversity by applying transformations like rotations, flips, and shifts. These transformations enforce invariance to position and orientation, which is desirable for object recognition. As a result, the model generalizes better to unseen test images.


In [4]:
datagen = ImageDataGenerator(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
                         zoom_range=0.1, horizontal_flip=True,
                         fill_mode="nearest")

validgen = ImageDataGenerator()

## Step 4: VGG-Inspired Model

This model is a VGG-inspired convolutional neural network consisting of stacked 3×3 convolutional layers with ReLU activations and max-pooling for hierarchical feature extraction. It incorporates Batch Normalization, 1×1 convolutions, Dropout, and Global Average Pooling to improve training stability and reduce overfitting. The network ends with fully connected layers and a softmax classifier for multi-class image classification.

In [5]:
# Model
in_shape = x_train[0].shape
model = models.Sequential()

# Filters per convolutional block
NEURONS = [64, 96, 160]  

# -------- Block 1: Low-level feature extraction --------
model.add(layers.Conv2D(filters = NEURONS[0], kernel_size = (3, 3), padding = 'same', input_shape=in_shape))
model.add(layers.Activation('relu'))
model.add(layers.Conv2D(filters = NEURONS[0], kernel_size = (3, 3), padding='same'))
model.add(layers.Activation('relu'))
model.add(layers.Conv2D(filters = NEURONS[0], kernel_size = (3, 3), padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPool2D(strides = (2, 2)))

# -------- Block 2: Mid-level feature extraction --------
model.add(layers.Conv2D(filters = NEURONS[1], kernel_size = (3, 3), padding='same'))
model.add(layers.Activation('relu'))
model.add(layers.Conv2D(filters = NEURONS[1], kernel_size = (3, 3), padding='same'))
model.add(layers.Activation('relu'))
model.add(layers.Conv2D(filters = NEURONS[1], kernel_size = (3, 3), padding='same'))
model.add(layers.BatchNormalization())
model.add(layers.MaxPool2D(strides = (2, 2)))

# -------- Block 3: Feature refinement --------
model.add(layers.Conv2D(filters = NEURONS[1], kernel_size = (3, 3), padding='same'))
model.add(layers.Activation('relu'))
model.add(layers.Conv2D(filters = NEURONS[1], kernel_size = (1, 1), padding='valid'))
model.add(layers.Activation('relu'))

# -------- Block 4: High-level feature compression --------
model.add(layers.Conv2D(filters = NEURONS[2], kernel_size = (1,1), padding='valid'))
model.add(layers.Dropout(rate = 0.3))
model.add(layers.GlobalAveragePooling2D())

# -------- Classification head --------
model.add(layers.Dense(units = 128, activation='relu'))
model.add(layers.Dropout(rate = 0.3))
model.add(layers.Dense(units = num_classes, activation='softmax'))

# Display model architecture
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 32, 64)        1792      
                                                                 
 activation (Activation)     (None, 32, 32, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 64)        36928     
                                                                 
 activation_1 (Activation)   (None, 32, 32, 64)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 32, 32, 64)        36928     
                                                                 
 batch_normalization (BatchN  (None, 32, 32, 64)       256       
 ormalization)                                                   
                                                        

2026-01-13 13:16:54.828958: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: /lib/x86_64-linux-gnu/libcuda.so.1: file too short; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2026-01-13 13:16:54.828973: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
# Loss function with label smoothing to reduce overconfidence and improve generalization
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# AdamW optimizer: Adam with decoupled weight decay for better regularization
optimizer = AdamW(learning_rate=3e-3, weight_decay=1e-4)

# Compile the model with optimizer, loss, and accuracy metric
model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# Training callbacks for adaptive learning and early stopping
callbacks = [ReduceLROnPlateau(
        monitor="loss",       # Reduce LR when training loss plateaus
        patience=4,
        factor=0.3,
        verbose=1,
        min_lr=1e-10
    ),
    EarlyStopping(
        monitor="loss",       # Stop training if no improvement
        patience=10,
        restore_best_weights=True
    )
]

## Step 5 : Training the model

In [ ]:
history = model.fit(
    datagen.flow(x_train, y_train, batch_size=64),
    validation_data = validgen.flow(x_test, y_test, batch_size=16),
    steps_per_epoch=len(x_train) // 64,
    validation_steps=len(x_test) // 16,
    epochs=200,
    callbacks=callbacks
)

# Plot Training & Validation Accuracy / Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(acc) + 1)

# Plot Accuracy
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs, acc, 'b-', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r--', label='Validation Accuracy')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
# Plot Loss
plt.subplot(1,2,2)
plt.plot(epochs, loss, 'b-', label='Training Loss')
plt.plot(epochs, val_loss, 'r--', label='Validation Loss')
plt.title('Training & Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


Epoch 1/200
781/781 [==============================] - 68s 85ms/step - loss: 1.9041 - accuracy: 0.3296 - val_loss: 1.7971 - val_accuracy: 0.4078 - lr: 0.0030
Epoch 2/200
781/781 [==============================] - 124s 159ms/step - loss: 1.6247 - accuracy: 0.4848 - val_loss: 1.8267 - val_accuracy: 0.4345 - lr: 0.0030
Epoch 3/200
781/781 [==============================] - 151s 193ms/step - loss: 1.4792 - accuracy: 0.5635 - val_loss: 1.4019 - val_accuracy: 0.5973 - lr: 0.0030
Epoch 4/200
781/781 [==============================] - 165s 211ms/step - loss: 1.3883 - accuracy: 0.6060 - val_loss: 1.4220 - val_accuracy: 0.6005 - lr: 0.0030
Epoch 5/200
781/781 [==============================] - 166s 212ms/step - loss: 1.3068 - accuracy: 0.6481 - val_loss: 1.4042 - val_accuracy: 0.6188 - lr: 0.0030
Epoch 6/200
781/781 [==============================] - 177s 227ms/step - loss: 1.2457 - accuracy: 0.6776 - val_loss: 1.3732 - val_accuracy: 0.6231 - lr: 0.0030
Epoch 7/200
781/781 [=====================

## Step 6 : Evaluate Model on test data

In [ ]:
# Evaluate the model on test data
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")

# Predict on the whole test set
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# If y_test is one‑hot, convert to indices
if len(y_test.shape) > 1 and y_test.shape[-1] > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - CIFAR-10")
plt.tight_layout()
plt.show()

## Step 7: Save the model


In [ ]:
model.save('cifar10_bw_Bmodel.h5')

## Step 8: Visualization of the code


In [ ]:
# Loading the model
model = load_model('model-group_9.h5') 

# Pick random test images
num_images = 9
indices = np.random.choice(len(x_test), num_images, replace=False)
images = x_test[indices]
true_labels = y_test[indices]

# If labels are one‑hot, convert to class indices
if len(true_labels.shape) > 1 and true_labels.shape[-1] > 1:
    true_labels_idx = np.argmax(true_labels, axis=1)
else:
    true_labels_idx = true_labels

# Predict
predictions = model.predict(images, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
confidences = np.max(predictions, axis=1)

# Plot
plt.figure(figsize=(10, 10))
for i in range(num_images):
    plt.subplot(3, 3, i + 1)

    # Ensure proper color scaling for CIFAR-10 (0–1 floats or 0–255 ints both work)
    img = images[i]
    if img.dtype != np.uint8 and img.max() <= 1.0:
        plt.imshow(img)
    else:
        plt.imshow(img.astype(np.uint8))

    plt.axis("off")

    true_class = class_names[int(true_labels_idx[i])]
    pred_class = class_names[int(predicted_labels[i])]
    confidence = float(confidences[i]) * 100

    color = "green" if true_labels_idx[i] == predicted_labels[i] else "red"
    plt.title(
        f"Pred: {pred_class}\nTrue: {true_class}\nConf: {confidence:.1f}%",
        fontsize=9,
        color=color,
    )

# Put suptitle slightly higher so tight_layout does not cover it
plt.suptitle("CIFAR-10 Test Images — Model Predictions", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()
